# Lab: Format-Preserving Encryption in practice
**Cloud Encryption, Day I, Session III**

Work through the exercises in order; Exercise 7 is a homework.

* Every exercise has cells marked `# TODO`: write your code there, then run the **check** cell below it.
* Run the next cell first: it loads the helper library (a pure-Python AES-128, FF1, HMAC-based PRF, Luhn utilities)
  and runs a self-test against the official test vectors.
* Educational code only: it is slow and **not** constant-time. Never use it in production.

In [ ]:
# ===== Helper library (run me first) =====
# =====================================================================
#  fpelab.py -- helper library for the FPE lab (Cloud Encryption, Day I)
#  Pure Python 3, standard library only: runs in JupyterLite (browser),
#  plain Jupyter, or `python3`.  Educational code: NOT constant time,
#  NOT for production use.
# =====================================================================
import hmac, hashlib, random, time, datetime

# ---------------------------------------------------------------------
#  A PRF for our toy constructions: HMAC-SHA256, output as an integer
# ---------------------------------------------------------------------
def prf_int(key, *parts):
    """HMAC-SHA256(key, parts) as a 256-bit integer. parts: str/int/bytes."""
    msg = b"|".join(p if isinstance(p, bytes) else str(p).encode() for p in parts)
    return int.from_bytes(hmac.new(key, msg, hashlib.sha256).digest(), "big")

# ---------------------------------------------------------------------
#  AES-128 (encryption only; FF1 never needs the inverse cipher)
# ---------------------------------------------------------------------
_SBOX = [
0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16]

def _xtime(a):
    a <<= 1
    return (a ^ 0x1b) & 0xff if a & 0x100 else a

def _expand_key(key):
    assert len(key) == 16, "AES-128 only"
    w = [list(key[i:i+4]) for i in range(0, 16, 4)]
    rcon = 1
    for i in range(4, 44):
        t = list(w[i-1])
        if i % 4 == 0:
            t = t[1:] + t[:1]
            t = [_SBOX[b] for b in t]
            t[0] ^= rcon
            rcon = _xtime(rcon)
        w.append([w[i-4][j] ^ t[j] for j in range(4)])
    return [sum(w[4*r:4*r+4], []) for r in range(11)]

class AES128:
    def __init__(self, key):
        self.rk = _expand_key(bytes(key))
    def encrypt_block(self, block):
        s = [b ^ k for b, k in zip(block, self.rk[0])]
        for rnd in range(1, 11):
            s = [_SBOX[b] for b in s]                                  # SubBytes
            s = [s[(i + 4*(i % 4)) % 16] for i in range(16)]           # ShiftRows
            if rnd != 10:                                              # MixColumns
                t = []
                for c in range(4):
                    a = s[4*c:4*c+4]
                    x = a[0] ^ a[1] ^ a[2] ^ a[3]
                    t += [a[j] ^ x ^ _xtime(a[j] ^ a[(j+1) % 4]) for j in range(4)]
                s = t
            s = [b ^ k for b, k in zip(s, self.rk[rnd])]               # AddRoundKey
        return bytes(s)

# ---------------------------------------------------------------------
#  FF1 (NIST SP 800-38G), AES-128, radix <= 36, digits 0-9a-z
# ---------------------------------------------------------------------
_ALPH = "0123456789abcdefghijklmnopqrstuvwxyz"

def _num(x, radix):
    v = 0
    for ch in x:
        v = v * radix + _ALPH.index(ch)
    return v

def _str(v, radix, m):
    out = []
    for _ in range(m):
        v, d = divmod(v, radix)
        out.append(_ALPH[d])
    return "".join(reversed(out))

def _ff1_round_value(aes, P, T, i, B, radix, b, d):
    t = len(T)
    Q = T + bytes((-t - b - 1) % 16) + bytes([i]) + _num(B, radix).to_bytes(b, "big")
    R = bytes(16)
    data = P + Q
    for j in range(0, len(data), 16):                                  # CBC-MAC
        R = aes.encrypt_block(bytes(x ^ y for x, y in zip(R, data[j:j+16])))
    S = R
    j = 1
    while len(S) < d:
        S += aes.encrypt_block(bytes(x ^ y for x, y in zip(R, j.to_bytes(16, "big"))))
        j += 1
    return int.from_bytes(S[:d], "big")

def _ff1_setup(key, tweak, n, radix):
    u = n // 2; v = n - u
    # ceil(v * log2(radix)) without floating point (cf. the 2025 draft)
    b = ((radix ** v - 1).bit_length() + 7) // 8
    d = 4 * ((b + 3) // 4) + 4
    P = bytes([1, 2, 1]) + radix.to_bytes(3, "big") + bytes([10, u % 256]) \
        + n.to_bytes(4, "big") + len(tweak).to_bytes(4, "big")
    return AES128(key), u, v, b, d, P

def ff1_encrypt(key, tweak, x, radix=10):
    """FF1 encryption of the numeral string x (characters 0-9a-z)."""
    n = len(x)
    aes, u, v, b, d, P = _ff1_setup(key, tweak, n, radix)
    A, B = x[:u], x[u:]
    for i in range(10):
        m = u if i % 2 == 0 else v
        y = _ff1_round_value(aes, P, tweak, i, B, radix, b, d)
        C = _str((_num(A, radix) + y) % radix ** m, radix, m)
        A, B = B, C
    return A + B

def ff1_decrypt(key, tweak, x, radix=10):
    n = len(x)
    aes, u, v, b, d, P = _ff1_setup(key, tweak, n, radix)
    A, B = x[:u], x[u:]
    for i in reversed(range(10)):
        m = u if i % 2 == 0 else v
        y = _ff1_round_value(aes, P, tweak, i, A, radix, b, d)
        C = _str((_num(B, radix) - y) % radix ** m, radix, m)
        B, A = A, C
    return A + B

# ---------------------------------------------------------------------
#  Small utilities
# ---------------------------------------------------------------------
def luhn_check_digit(body):
    """Luhn check digit for the digit string `body` (all digits but the last)."""
    total = 0
    for i, ch in enumerate(reversed(body)):
        dgt = int(ch)
        if i % 2 == 0:                    # doubled positions (rightmost of the body)
            dgt *= 2
            if dgt > 9:
                dgt -= 9
        total += dgt
    return str((10 - total % 10) % 10)

def luhn_ok(number):
    return luhn_check_digit(number[:-1]) == number[-1]

def random_key():
    return bytes(random.getrandbits(8) for _ in range(16))

def timeit(f, reps=200):
    """Average running time of f() in microseconds."""
    t0 = time.perf_counter()
    for _ in range(reps):
        f()
    return 1e6 * (time.perf_counter() - t0) / reps

# ---------------------------------------------------------------------
#  Self-test (FIPS-197 and SP 800-38G sample vectors)
# ---------------------------------------------------------------------
# Source: NIST, FF1 examples with intermediate values (SP 800-38G), samples 1-3 (AES-128):
# https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/FF1samples.pdf
# AES vector: FIPS-197, Appendix C.1
FF1_SAMPLES = [  # (key, tweak, radix, plaintext, ciphertext): NIST FF1 samples 1-3
    ("2b7e151628aed2a6abf7158809cf4f3c", "", 10, "0123456789", "2433477484"),
    ("2b7e151628aed2a6abf7158809cf4f3c", "39383736353433323130", 10, "0123456789", "6124200773"),
    ("2b7e151628aed2a6abf7158809cf4f3c", "3737373770717273373737", 36,
     "0123456789abcdefghi", "a9tv40mll9kdu509eum"),
]

def selftest():
    aes = AES128(bytes.fromhex("000102030405060708090a0b0c0d0e0f"))
    assert aes.encrypt_block(bytes.fromhex("00112233445566778899aabbccddeeff")).hex() \
        == "69c4e0d86a7b0430d8cdb78070b4c55a", "AES self-test failed"
    for k, t, r, pt, ct in FF1_SAMPLES:
        k, t = bytes.fromhex(k), bytes.fromhex(t)
        assert ff1_encrypt(k, t, pt, r) == ct, "FF1 sample failed"
        assert ff1_decrypt(k, t, ct, r) == pt, "FF1 decryption failed"
    print("Self-test OK: AES-128 (FIPS-197) and FF1 (NIST samples 1-3)")


selftest()

## Exercise 1 — A Feistel network on digit strings
Implement the alternating Feistel network used by FF1, on a string `x` of $n\ge 2$ decimal digits:

* split $x = A\,\|\,B$ with $|A| = u = \lfloor n/2\rfloor$, $|B| = v = n-u$;
* in round $i$ let $m = u$ if $i$ is even, $m=v$ if $i$ is odd, compute $y = \mathrm{PRF}_K(T, i, B)$ with `prf_int(key, tweak, i, B)`,
  then $C = (A + y) \bmod 10^m$ written with exactly $m$ digits, and set $A, B \leftarrow B, C$;
* after the last round output $A\,\|\,B$.

Then write the inverse. *Hint:* in the inverse you know $A$ (= the old $B$), so you can recompute $y$.

In [ ]:
def feistel_encrypt(key, tweak, x, rounds=10):
    raise NotImplementedError  # TODO

def feistel_decrypt(key, tweak, x, rounds=10):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
key = b"lab-key-1"
for x in ["12", "123", "123456", "4385822056110988"]:
    c = feistel_encrypt(key, b"T1", x)
    assert len(c) == len(x) and c.isdigit() and feistel_decrypt(key, b"T1", c) == x
    print(x, "->", c)

# Is it a permutation of {0000, ..., 9999}?  (brute force over all 10^4 inputs)
images = {feistel_encrypt(key, b"T1", f"{i:04d}") for i in range(10**4)}
print("distinct images:", len(images), "(expected 10000)")

# A different tweak gives a different-looking permutation
diff = sum(feistel_encrypt(key, b"T1", f"{i:04d}") != feistel_encrypt(key, b"T2", f"{i:04d}") for i in range(10**4))
print("inputs encrypted differently under T1 and T2:", diff,
      "(about 9999: two random permutations agree on 1 point on average, so a handful of agreements is normal)")

**Question.** Why is the network a permutation *whatever* `prf_int` returns? Which property of the round function
did you never use?

## Exercise 2 — FF1 on the hospital table
The library contains a real FF1 (`ff1_encrypt`, `ff1_decrypt`), checked against NIST's published FF1 examples
(samples 1–3, AES-128; https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/FF1samples.pdf).
Each line shows the library's output next to NIST's value: they must match.

In [ ]:
for k, t, r, pt, ct in FF1_SAMPLES:
    print(f"radix {r:2}: FF1({pt}) = {ff1_encrypt(bytes.fromhex(k), bytes.fromhex(t), pt, r)}   (NIST: {ct})")

A card number has 16 digits and the last one is a **Luhn check digit**, so the format is
$\{x \in \{0..9\}^{16} : \text{Luhn}(x)\}$, not $\{0..9\}^{16}$. Use **rank–encipher–unrank**:

* **rank:** keep the first 15 digits (the check digit is determined by them);
* **encipher:** FF1 on those 15 digits;
* **unrank:** append the Luhn digit of the result (`luhn_check_digit`).

*Python notes.* `tweak=b"card"`: one fixed tweak for the whole column (the bytes of the word *card*; other columns
would use other labels). Later, `rid.encode()` turns the record id into bytes, e.g. `"r001"` → `b"r001"`: one tweak per record.

In [ ]:
def encrypt_card(key, card, tweak=b"card"):
    raise NotImplementedError  # TODO

def decrypt_card(key, card, tweak=b"card"):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check: encrypt the patients table -----
patients = [  # (record id, name, card number, birth date)
    ("r001", "M. Rossi",   "4385822056110988", "1983-02-14"),
    ("r002", "L. Bianchi", "4485392056112839", "1990-07-01"),
    ("r003", "G. Verdi",   "5354128799045612", "1975-11-30"),
    ("r004", "M. Rossi",   "4385822056110988", "1983-02-14"),   # second visit, same card
]
key = random_key()
for rid, name, card, bdate in patients:
    c = encrypt_card(key, card)
    assert luhn_ok(c) and decrypt_card(key, c) == card
    print(f"{rid}  {name:11} {card} -> {c}")

**Questions.** (a) Rows `r001` and `r004` get the *same* ciphertext: which security notion of Session I explains why this
is unavoidable? What does it leak to the cloud provider?
(b) Re-run the encryption with `tweak = rid.encode()` (one tweak per record). What changes? What do we lose
(think of a `JOIN` or a search on the card column)?

In [ ]:
# TODO (b): encrypt with a per-record tweak and compare

## Exercise 3 — Few rounds are dangerous
Take your `feistel_encrypt` on **4 digits** ($u=v=2$) with only **2 rounds**. Write a distinguisher that gets an
oracle `enc` (either 2-round Feistel with a random key, or a truly random permutation), makes **two queries** and
outputs 1 if it believes it is talking to Feistel.

*Hint:* query $x_1 = 00\|00$ and $x_2 = 01\|00$ (same right half). After two rounds the left half of the output is
$C_0 = A + F_0(B) \bmod 100$. What is $C_0(x_2) - C_0(x_1) \bmod 100$?

In [ ]:
def distinguisher(enc):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check: estimate the advantage -----
def lazy_random_perm(ndigits=4):
    """A uniformly random permutation of the ndigits-digit strings, sampled lazily."""
    table, used = {}, set()
    def enc(x):
        if x not in table:
            while True:
                y = f"{random.randrange(10**ndigits):0{ndigits}d}"
                if y not in used:
                    break
            table[x] = y; used.add(y)
        return table[x]
    return enc

def advantage(rounds, trials=300):
    hits_feistel = sum(distinguisher(lambda x, k=random_key(): feistel_encrypt(k, b"", x, rounds))
                       for _ in range(trials))
    hits_random = sum(distinguisher(lazy_random_perm()) for _ in range(trials))
    return (hits_feistel - hits_random) / trials

for r in [2, 3, 4, 10]:
    print(f"{r:2} rounds: estimated advantage {advantage(r):+.2f}")

**Questions.** Why does **this** attack die at 3 rounds? What does Luby–Rackoff (Session II) guarantee on a
half-domain of only 100 values?
**Homework:** with *inverse* queries, 3 rounds can still be broken: find the attack.

## Exercise 4 — The FF3 slide attack, in code
Below is a **toy FF3**: 8 rounds on 4 digits; round $i$ uses the tweak half $W_i = T_R$ ($i$ even) or $T_L$ ($i$ odd)
and feeds $W_i \oplus i$ to the round function (exactly the FF3 tweak schedule; the round function is our HMAC PRF).

Session II claims: with $T' = (T_L\oplus 4,\ T_R\oplus 4)$ we get $\mathrm{FF3}_{T} = G\circ F$ and $\mathrm{FF3}_{T'} = F\circ G$,
hence $\mathrm{FF3}_{T'} = F\circ \mathrm{FF3}_T\circ F^{-1}$: the two permutations are **conjugate** and have the same
**cycle structure**. Let's check it.

In [ ]:
def toy_ff3_rounds(key, TL, TR, x, start, stop):
    """Rounds start..stop-1 of the toy FF3 on a 4-digit string x."""
    A, B = x[:2], x[2:]
    for i in range(start, stop):
        W = TR if i % 2 == 0 else TL
        y = prf_int(key, W ^ i, B)
        A, B = B, str((int(A) + y) % 100).zfill(2)
    return A + B

def toy_ff3(key, TL, TR, x):
    return toy_ff3_rounds(key, TL, TR, x, 0, 8)

**(a)** Write `cycle_type(f)`: the sorted list of cycle lengths of the permutation $x\mapsto f(x)$ on the
4-digit strings. **(b)** Compare the cycle types for $T$, $T'$ and a random $T''$.
**(c)** Check the slide equation $\mathrm{FF3}_{T'}(F(x)) = F(\mathrm{FF3}_T(x))$ for **all** $x$,
where $F$ = rounds 0–3 under $T$.

In [ ]:
def cycle_type(f, ndigits=4):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
key = random_key()
TL, TR = random.getrandbits(32), random.getrandbits(32)
ct_T   = cycle_type(lambda x: toy_ff3(key, TL, TR, x))
ct_T1  = cycle_type(lambda x: toy_ff3(key, TL ^ 4, TR ^ 4, x))
TL2, TR2 = random.getrandbits(32), random.getrandbits(32)          # an unrelated tweak T''
ct_T2  = cycle_type(lambda x: toy_ff3(key, TL2, TR2, x))
print("longest cycles under T  :", ct_T[:6])
print("longest cycles under T' :", ct_T1[:6])
print("longest cycles under T'':", ct_T2[:6])
print("T and T' have the same cycle type: ", ct_T == ct_T1)
print("T and T'' have the same cycle type:", ct_T == ct_T2)

In [ ]:
# TODO (c): count the x in {0000..9999} with toy_ff3(key, TL^4, TR^4, F(x)) == F(toy_ff3(key, TL, TR, x))

**Questions.** (a) Why does a *unique* longest cycle give the attacker pairs $(x, F(x))$ for free?
(b) FF3-1 builds each tweak half as `(28 tweak bits) || 0000`. Why does this kill the attack?
(c) The attack never touches AES. What does that tell you about "FF3 uses AES, so it is secure"?

## Exercise 5 — Birth dates with cycle walking
Dates between 1900-01-01 and 2099-12-31 can be ranked as day numbers in $\Z_M$ with $M = 73049$, which is not a power of 10.
Encrypt the rank with FF1 on **5 digits** ($10^5$ values) and **cycle-walk** until the result is $< M$.
Measure the average number of FF1 calls and compare with $\frac{|\mathcal X|+1}{|\mathcal Y|+1}$.

In [ ]:
D0 = datetime.date(1900, 1, 1)
NDAYS = (datetime.date(2099, 12, 31) - D0).days + 1     # 73049

def encrypt_date(key, d, tweak=b"birthdate"):
    """Return (ciphertext date, number of FF1 calls)."""
    raise NotImplementedError  # TODO

def decrypt_date(key, d, tweak=b"birthdate"):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
key = random_key()
d = datetime.date(1983, 2, 14)
c, calls = encrypt_date(key, d)
assert decrypt_date(key, c) == d
print(d, "->", c, f"({calls} FF1 calls)")
counts = [encrypt_date(key, D0 + datetime.timedelta(days=random.randrange(NDAYS)))[1] for _ in range(300)]
print("average calls:", sum(counts) / len(counts), " expected:", (10**5 + 1) / (NDAYS + 1))
print("max calls:", max(counts))

**Question.** The number of calls depends on the date. Can a timing measurement leak something?

## Exercise 6 — Swap-or-Not
Implement Swap-or-Not on $\Z_N$: for $i = 1..r$, $X' = K_i - X \bmod N$, $\hat X = \max(X, X')$,
and if $F_i(\hat X) = 1$ then $X \leftarrow X'$. Use `K[i] = prf_int(key, "K", i) % N` and
`F_i(z) = prf_int(key, "F", i, z) & 1`. Decryption runs the same rounds in reverse order.

In [ ]:
def swap_or_not(key, x, N, rounds, inverse=False):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check (N = 1000; 469 rounds give Adv^cca_SN <= 2^-40 at q = N/2 by the HMR'12 bound) -----
key, N, r = random_key(), 1000, 469
imgs = [swap_or_not(key, x, N, r) for x in range(N)]
assert sorted(imgs) == list(range(N)), "not a permutation!"
assert all(swap_or_not(key, y, N, r, inverse=True) == x for x, y in enumerate(imgs[:50]))
print("Swap-or-Not is a permutation of Z_1000 and decrypts correctly")
print(f"Swap-or-Not ({r} rounds): {timeit(lambda: swap_or_not(key, 123, N, r), 20):8.0f} us per encryption")
print(f"FF1 (10 rounds, AES)  : {timeit(lambda: ff1_encrypt(key, b'', '123'), 20):8.0f} us per encryption")

**Note.** Our AES is pure Python, so FF1 looks slow here; with hardware AES an FF1 call costs about a microsecond.
Compare the number of **PRF calls** instead: 469 for Swap-or-Not vs. about 20 AES calls for FF1.

## Exercise 7 (homework) — A simplified *codice fiscale*
Format (we ignore the final check character and the validity of names, dates and municipalities):
6 letters, 2 digits (year), 1 month letter from `ABCDEHLMPRST`, 2 digits for the day (`01`–`31` or `41`–`71`),
1 letter and 3 digits (municipality code). E.g. `RSSMRA80A41H501`.

1. Write a **mixed-radix** `rank` / `unrank` between this format and $\Z_M$ (compute $M$).
2. Encrypt with FF1 on $\lceil\log_{10} M\rceil$ digits plus cycle walking (as in Exercise 5).
3. What would you need to also keep the check character valid?

In [ ]:
LET, MON = "ABCDEFGHIJKLMNOPQRSTUVWXYZ", "ABCDEHLMPRST"
DAYS = [f"{d:02d}" for d in list(range(1, 32)) + list(range(41, 72))]   # 62 values
FIELDS = [LET]*6 + ["0123456789"]*2 + [MON] + [DAYS] + [LET] + ["0123456789"]*3

def cf_tokens(s):      # split the string into the 14 fields above
    return list(s[:9]) + [s[9:11]] + list(s[11:])

def cf_rank(s):
    raise NotImplementedError  # TODO

def cf_unrank(r):
    raise NotImplementedError  # TODO